#CONNECTION TO GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

os.listdir('/content/drive/MyDrive/Bluestock /capstone')

['raw', 'cleaned', 'notebook']

In [ ]:
raw_path = "/content/drive/MyDrive/Bluestock /capstone/raw"

print(os.listdir(raw_path))

['1788499980509-304c1255-08_investor_transactions.csv', '1788499982117-e3d6ab98-09_portfolio_holdings.csv', '1788499982615-f9647ab2-10_benchmark_indices.csv', '1788499983024-b042c300-01_fund_master.csv', '1788499983331-4389156d-02_nav_history.csv', '1788499984405-d702a6c6-04_monthly_sip_inflows.csv', '1788499984721-4b860901-05_category_inflows.csv', '1788499985036-da4a0c4a-06_industry_folio_count.csv', '1788499985420-bb134abf-07_scheme_performance.csv', '1788499984134-b0cbf625-03_aum_by_fund_house.csv']


In [ ]:
cleaned_path = "/content/drive/MyDrive/Bluestock /capstone/cleaned"

print(os.listdir(cleaned_path))

['fund_master_cleaned.csv', 'fund_holdings_cleaned.csv', 'monthly_SIP_inflows_cleaned.csv', 'industry_folio_count_cleaned.csv', 'scheme_performance_cleaned.csv', 'Category_inflow_cleaned.csv', 'aum_cleaned.csv', 'nav_history_cleaned.csv', 'investor_transactions_cleaned.csv', 'Benchmark indices cleaned.csv']


#PART 1 DATASETS CLEANING

# Data Cleaning – Investor Transactions

In [ ]:
import pandas as pd

df_transactions = pd.read_csv(
    os.path.join(raw_path, "1788499980509-304c1255-08_investor_transactions.csv")
)

df_transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [ ]:
#data inspection
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [ ]:
df_transactions["transaction_type"].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [ ]:
df_transactions["transaction_type"] = (
    df_transactions["transaction_type"]
    .astype(str)
    .str.strip()
)

In [ ]:
(df_transactions["amount_inr"] <= 0).sum()

np.int64(0)

In [ ]:
df_transactions["transaction_date"].head()

,transaction_date
0,2024-01-01
1,2024-01-01
2,2024-01-01
3,2024-01-01
4,2024-01-01


In [ ]:
df_transactions["transaction_date"] = pd.to_datetime(
    df_transactions["transaction_date"],
    errors="coerce"
)

In [ ]:
df_transactions["kyc_status"].unique()

array(['Verified', 'Pending'], dtype=object)

In [ ]:
valid_kyc_status = ["Verified", "Pending"]

invalid_kyc = df_transactions[
    ~df_transactions["kyc_status"].isin(valid_kyc_status)
]

len(invalid_kyc)

0

In [ ]:
df_transactions.to_csv(
    os.path.join(cleaned_path, "investor_transactions_cleaned.csv"),
    index=False
)

#Data Cleaning – NAV History

In [ ]:
df_nav = pd.read_csv(
    os.path.join(raw_path, "1788499983331-4389156d-02_nav_history.csv")
)

df_nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [ ]:
df_nav["date"] = pd.to_datetime(
    df_nav["date"],
    dayfirst=True,
    errors="coerce"
)

In [ ]:
df_nav = df_nav.sort_values(
    by=["amfi_code", "date"]
).reset_index(drop=True)

In [ ]:
df_nav["nav"] = df_nav.groupby("amfi_code")["nav"].ffill()

In [ ]:
df_nav = df_nav.drop_duplicates()

In [ ]:
(df_nav["nav"] <= 0).sum()

np.int64(0)

In [ ]:
df_nav.to_csv(
    os.path.join(cleaned_path, "nav_history_cleaned.csv"),
    index=False
)

#Data Cleaning – Scheme Performance

In [ ]:
df_performance = pd.read_csv(
    os.path.join(raw_path, "1788499985420-bb134abf-07_scheme_performance.csv")
)

df_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [ ]:
return_columns = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct"
]

df_performance[return_columns].dtypes

,0
return_1yr_pct,float64
return_3yr_pct,float64
return_5yr_pct,float64
benchmark_3yr_pct,float64


In [ ]:
df_performance[return_columns].describe()

,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct
count,40.000000,40.000000,40.000000,40.000000
mean,14.376000,14.089000,14.516750,12.835500
std,4.883023,4.617253,4.454021,4.740972
min,4.260000,5.140000,5.430000,3.960000
25%,11.735000,12.035000,12.340000,10.690000
50%,14.620000,14.205000,14.185000,13.090000
75%,16.392500,15.882500,17.585000,14.775000
max,24.930000,23.390000,23.800000,22.160000


In [ ]:
for column in return_columns:
    Q1 = df_performance[column].quantile(0.25)
    Q3 = df_performance[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    anomalies = df_performance[
        (df_performance[column] < lower_bound) |
        (df_performance[column] > upper_bound)
    ]

    print(column, "anomalies:", len(anomalies))

return_1yr_pct anomalies: 3
return_3yr_pct anomalies: 7
return_5yr_pct anomalies: 0
benchmark_3yr_pct anomalies: 5


In [ ]:
invalid_expense = df_performance[
    (df_performance["expense_ratio_pct"] < 0.1) |
    (df_performance["expense_ratio_pct"] > 2.5)
]

len(invalid_expense)

0

In [ ]:
df_performance.to_csv(
    os.path.join(cleaned_path, "scheme_performance_cleaned.csv"),
    index=False
)

#Data Cleaning – AUM Dataset

In [ ]:
df_aum = pd.read_csv(
    os.path.join(raw_path, "1788499984134-b0cbf625-03_aum_by_fund_house.csv")
)

df_aum.head()

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [ ]:
df_aum["date"] = pd.to_datetime(
    df_aum["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
df_aum.duplicated().sum()

np.int64(0)

In [ ]:
#Numeric data validation
df_aum[
    ["aum_lakh_crore", "aum_crore", "num_schemes"]
].dtypes

,0
aum_lakh_crore,float64
aum_crore,int64
num_schemes,int64


In [ ]:
(df_aum[["aum_lakh_crore", "aum_crore", "num_schemes"]] <= 0).sum()

,0
aum_lakh_crore,0
aum_crore,0
num_schemes,0


In [ ]:
df_aum.to_csv(
    os.path.join(cleaned_path, "aum_cleaned.csv"),
    index=False
)

# Data cleaning - Category inflows

In [ ]:
df_inflow = pd.read_csv(
    os.path.join(raw_path, "1788499984721-4b860901-05_category_inflows.csv")
)

df_inflow.head()

,month,category,net_inflow_crore
0,2024-04,Large Cap,2413.0
1,2024-04,Mid Cap,3897.0
2,2024-04,Small Cap,3533.0
3,2024-04,Flexi Cap,4947.0
4,2024-04,Large & Mid Cap,4214.0


In [ ]:
df_inflow["month"] = pd.to_datetime(
    df_inflow["month"],
    format="%Y-%m",
    errors="coerce"
)

In [ ]:
df_inflow.duplicated().sum()

np.int64(0)

In [ ]:
df_inflow["net_inflow_crore"].dtype

dtype('float64')

In [ ]:
df_inflow.isnull().sum()

,0
month,0
category,0
net_inflow_crore,0


In [ ]:
df_inflow.to_csv(
    os.path.join(cleaned_path, "Category_inflow_cleaned.csv"),
    index=False
)

#Data cleaning - Industry folio count

In [ ]:
df_folios = pd.read_csv(
    os.path.join(raw_path, "1788499985036-da4a0c4a-06_industry_folio_count.csv")
)

df_folios.head()

,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01,13.26,9.28,1.86,0.80,1.33
1,2022-04,13.91,9.74,1.95,0.83,1.39
2,2022-07,13.85,9.69,1.94,0.83,1.38
3,2022-10,14.12,9.88,1.98,0.85,1.41
4,2023-01,14.81,10.37,2.07,0.89,1.48


In [ ]:
df_folios["month"] = pd.to_datetime(
    df_folios["month"],
    format="%Y-%m",
    errors="coerce"
)

In [ ]:
df_folios.duplicated().sum()

np.int64(0)

In [ ]:
folio_columns = [
    "total_folios_crore",
    "equity_folios_crore",
    "debt_folios_crore",
    "hybrid_folios_crore",
    "others_folios_crore"
]

df_folios[folio_columns].dtypes

,0
total_folios_crore,float64
equity_folios_crore,float64
debt_folios_crore,float64
hybrid_folios_crore,float64
others_folios_crore,float64


In [ ]:
(df_folios[folio_columns] < 0).sum()

,0
total_folios_crore,0
equity_folios_crore,0
debt_folios_crore,0
hybrid_folios_crore,0
others_folios_crore,0


In [ ]:
df_folios.isnull().sum()

,0
month,0
total_folios_crore,0
equity_folios_crore,0
debt_folios_crore,0
hybrid_folios_crore,0
others_folios_crore,0


In [ ]:
df_folios.to_csv(
    os.path.join(cleaned_path, "industry_folio_count_cleaned.csv"),
    index=False
)

#Data cleaning - Benchmark indices

In [ ]:
df_index = pd.read_csv(
    os.path.join(raw_path, "1788499982615-f9647ab2-10_benchmark_indices.csv")
)

df_index.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [ ]:
df_index["date"] = pd.to_datetime(
    df_index["date"],
    dayfirst=True,
    errors="coerce"
)

In [ ]:
duplicate_rows = df_index[df_index.duplicated(keep=False)]

duplicate_rows

,date,index_name,close_value
815,NaT,NIFTY50,20464.36
909,NaT,NIFTY50,20464.36
5953,NaT,CRISIL_LIQUID,2403.76
5954,NaT,CRISIL_LIQUID,2403.76
6066,NaT,CRISIL_LIQUID,2460.70
6068,NaT,CRISIL_LIQUID,2460.70
6177,NaT,CRISIL_LIQUID,2541.20
6179,NaT,CRISIL_LIQUID,2541.20
6197,NaT,CRISIL_LIQUID,2548.81
6202,NaT,CRISIL_LIQUID,2548.81


In [ ]:
df_index_original = pd.read_csv(
    os.path.join(raw_path, "1788499982615-f9647ab2-10_benchmark_indices.csv")
)

df_index_original.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [ ]:
df_index = df_index_original.copy()

df_index["date"] = pd.to_datetime(
    df_index["date"],
    dayfirst=True,
    errors="coerce"
)

In [ ]:
df_index["date"].isna().sum()

np.int64(4879)

In [ ]:
df_index.duplicated().sum()

np.int64(12)

In [ ]:
df_index["close_value"].dtype

dtype('float64')

In [ ]:
(df_index["close_value"]<=0).sum()

np.int64(0)

In [ ]:
df_index.to_csv(
    os.path.join(cleaned_path, "Benchmark indices cleaned.csv"),
    index=False
)

#Data cleaning - Fund master

In [ ]:
df_fund = pd.read_csv(
    os.path.join(raw_path, "1788499983024-b042c300-01_fund_master.csv")
)

df_fund.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [ ]:
df_fund["launch_date"] = pd.to_datetime(
    df_fund["launch_date"],
    dayfirst=True,
    errors="coerce"
)

/tmp/ipykernel_524/2365658674.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_fund["launch_date"] = pd.to_datetime(


In [ ]:
df_fund["launch_date"] = pd.to_datetime(
    df_fund["launch_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [ ]:
df_fund["launch_date"].isna().sum()

In [ ]:
df_fund.duplicated().sum()

In [ ]:
numeric_columns = [
    "expense_ratio_pct",
    "exit_load_pct",
    "min_sip_amount",
    "min_lumpsum_amount"
]

df_fund[numeric_columns].dtypes

In [ ]:
(df_fund[numeric_columns]<=0).sum()

In [ ]:
df_fund[df_fund["exit_load_pct"] < 0]

In [ ]:
df_fund["exit_load_pct"].value_counts().sort_index()

In [ ]:
df_fund.isnull().sum()

In [ ]:
df_fund.to_csv(
    os.path.join(cleaned_path, "fund_master_cleaned.csv"),
    index=False
)

#Data cleaning - Portfolio holdings

In [ ]:
df_holdings = pd.read_csv(
    os.path.join(raw_path, "1788499982117-e3d6ab98-09_portfolio_holdings.csv")
)

df_holdings.head()

,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31


In [ ]:
df_holdings["portfolio_date"] = pd.to_datetime(
    df_holdings["portfolio_date"],
    dayfirst=True,
    errors="coerce"
)

In [ ]:
df_holdings["portfolio_date"] = pd.to_datetime(
    df_holdings["portfolio_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [ ]:
df_holdings.duplicated().sum()

In [ ]:
holding_numeric_columns = [
    "weight_pct",
    "market_value_cr",
    "current_price_inr"
]

df_holdings[holding_numeric_columns].dtypes

In [ ]:
(df_holdings[holding_numeric_columns] <= 0).sum()

In [ ]:
df_holdings.isnull().sum()

In [ ]:
df_holdings.to_csv(
    os.path.join(cleaned_path, "fund_holdings_cleaned.csv"),
    index=False
)

#Data cleaning - Monthly SIP inflows

In [ ]:
df_monthlySIP = pd.read_csv(
    os.path.join(raw_path, "1788499984405-d702a6c6-04_monthly_sip_inflows.csv")
)

df_monthlySIP.head()

,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01,11517,4.91,9.10,4.80,NaN
1,2022-02,11438,4.93,8.20,4.85,NaN
2,2022-03,12328,5.09,10.50,5.01,NaN
3,2022-04,11863,5.48,9.52,5.12,NaN
4,2022-05,12286,5.55,8.10,5.15,NaN


In [ ]:
df_monthlySIP.columns = df_monthlySIP.columns.str.strip()

In [ ]:
df_monthlySIP.head()



In [ ]:
df_monthlySIP["month"] = pd.to_datetime(
    df_monthlySIP["month"],
    format="%y-%m",
    errors="coerce"
)

In [ ]:
numeric_columns = [
    "sip_inflow_crore",
    "active_sip_accounts_crore",
    "new_sip_accounts_lakh",
    "sip_aum_lakh_crore",
    "yoy_growth_pct"
]

df_monthlySIP[numeric_columns].dtypes

In [ ]:
(df_monthlySIP[numeric_columns] <= 0).sum()

In [ ]:
df_monthlySIP.isnull().sum()

In [ ]:
df_monthlySIP["month"] = pd.to_datetime(
    df_monthlySIP["month"],
    errors="coerce"
)

In [ ]:
df_monthlySIP.isnull().sum()

In [ ]:
df_monthlySIP = pd.read_csv("1788499984405-d702a6c6-04_monthly_sip_inflows.csv")

In [ ]:
print(df_monthlySIP["month"].head(10))

In [ ]:
df_monthlySIP["month"] = pd.to_datetime(
    df_monthlySIP["month"].astype(str).str.strip(),
    format="%Y-%m",
    errors="coerce"
)

In [ ]:
print(df_monthlySIP["month"].head())
print(df_monthlySIP["month"].isnull().sum())

In [ ]:
df_monthlySIP.duplicated().sum()

In [ ]:
df_monthlySIP.isnull().sum()

In [ ]:
df_monthlySIP.to_csv(
    os.path.join(cleaned_path, "monthly_SIP_inflows_cleaned.csv"),
    index=False
)

#PART 2 SQLITE DATABASE

# SQLite Database Setup

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine("sqlite:///bluestock_mf.db")

In [ ]:
with engine.connect() as connection:
    print("SQLite database connected successfully!")

SQLite database connected successfully!


# SQLite Star Schema Design

In [ ]:
print("TRANSACTIONS:", df_transactions.columns.tolist())
print("\nNAV:", df_nav.columns.tolist())
print("\nFUND MASTER:", df_fund.columns.tolist())
print("\nPERFORMANCE:", df_performance.columns.tolist())
print("\nAUM:", df_aum.columns.tolist())

TRANSACTIONS: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

NAV: ['amfi_code', 'date', 'nav']

FUND MASTER: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

PERFORMANCE: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

AUM: ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']


In [ ]:
from sqlalchemy import text

schema_sql = """
DROP TABLE IF EXISTS fact_nav;
DROP TABLE IF EXISTS fact_transactions;
DROP TABLE IF EXISTS fact_performance;
DROP TABLE IF EXISTS fact_aum;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_fund;

CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date TEXT,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER
);

CREATE TABLE fact_nav (
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    PRIMARY KEY (amfi_code, date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE fact_transactions (
    investor_id INTEGER,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE fact_performance (
    amfi_code INTEGER PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating REAL,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_aum (
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER,
    PRIMARY KEY (date, fund_house),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);
"""

with engine.begin() as conn:
    for statement in schema_sql.split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("Star schema tables created successfully!")

Star schema tables created successfully!


In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

print("Tables created:")
print(inspector.get_table_names())

Tables created:
['dim_date', 'dim_fund', 'fact_aum', 'fact_nav', 'fact_performance', 'fact_transactions']


In [ ]:
df_fund.to_sql(
    "dim_fund",
    con=engine,
    if_exists="append",
    index=False
)

print("dim_fund loaded successfully!")

dim_fund loaded successfully!


In [ ]:
# Create a combined list of all unique dates
all_dates = pd.concat([
    pd.to_datetime(df_nav["date"]),
    pd.to_datetime(df_transactions["transaction_date"]),
    pd.to_datetime(df_aum["date"])
]).dropna().drop_duplicates()

# Create dim_date
dim_date = pd.DataFrame({
    "date": all_dates.dt.strftime("%Y-%m-%d"),
    "year": all_dates.dt.year,
    "month": all_dates.dt.month,
    "day": all_dates.dt.day
})

# Remove any duplicate dates
dim_date = dim_date.drop_duplicates(subset=["date"])

# Load into SQLite
dim_date.to_sql(
    "dim_date",
    con=engine,
    if_exists="append",
    index=False
)

print("dim_date loaded successfully!")
print("Number of dates:", len(dim_date))

dim_date loaded successfully!
Number of dates: 1297


In [ ]:
df_nav.to_sql(
    "fact_nav",
    con=engine,
    if_exists="append",
    index=False
)

print("fact_nav loaded successfully!")
print("Rows loaded:", len(df_nav))

fact_nav loaded successfully!
Rows loaded: 46000


In [ ]:
df_transactions.to_sql(
    "fact_transactions",
    con=engine,
    if_exists="append",
    index=False
)

print("fact_transactions loaded successfully!")
print("Rows loaded:", len(df_transactions))

fact_transactions loaded successfully!
Rows loaded: 32778


In [ ]:
performance_columns = [
    "amfi_code",
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating",
    "risk_grade"
]

df_performance_fact = df_performance[performance_columns]

df_performance_fact.to_sql(
    "fact_performance",
    con=engine,
    if_exists="append",
    index=False
)

print("fact_performance loaded successfully!")
print("Rows loaded:", len(df_performance_fact))

fact_performance loaded successfully!
Rows loaded: 40


In [ ]:
df_aum.to_sql(
    "fact_aum",
    con=engine,
    if_exists="append",
    index=False
)

print("fact_aum loaded successfully!")
print("Rows loaded:", len(df_aum))

fact_aum loaded successfully!
Rows loaded: 90


In [ ]:
from sqlalchemy import text

tables = [
    "dim_fund",
    "dim_date",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )
        count = result.scalar()
        print(f"{table}: {count} rows")

dim_fund: 40 rows
dim_date: 1297 rows
fact_nav: 46000 rows
fact_transactions: 32778 rows
fact_performance: 40 rows
fact_aum: 90 rows


In [ ]:
print("SOURCE DATAFRAME ROW COUNTS")
print("dim_fund source:", len(df_fund))
print("fact_nav source:", len(df_nav))
print("fact_transactions source:", len(df_transactions))
print("fact_performance source:", len(df_performance_fact))
print("fact_aum source:", len(df_aum))

SOURCE DATAFRAME ROW COUNTS
dim_fund source: 40
fact_nav source: 46000
fact_transactions source: 32778
fact_performance source: 40
fact_aum source: 90


#SQL Queries

#Query 1: Top 5 funds by AUM

In [ ]:
query1 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.aum_crore
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
"""

top_5_aum = pd.read_sql(query1, engine)

top_5_aum

,scheme_name,fund_house,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,49046.0
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,47469.0
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,43630.0
3,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,41828.0
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,41728.0


#Query 2 — Average NAV per month

In [ ]:
query2 = """
SELECT
    strftime('%Y-%m', date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY strftime('%Y-%m', date)
ORDER BY month;
"""

avg_nav_monthly = pd.read_sql(query2, engine)

avg_nav_monthly

,month,average_nav
0,2022-01,207.06
1,2022-02,207.72
2,2022-03,209.69
3,2022-04,211.83
4,2022-05,212.73
5,2022-06,213.86
6,2022-07,213.96
7,2022-08,215.68
8,2022-09,218.49
9,2022-10,219.53


#Query 3: SIP Year-over-Year (YoY) Growth

In [ ]:
query3 = """
WITH yearly_sip AS (
    SELECT
        strftime('%Y', transaction_date) AS year,
        SUM(amount_inr) AS total_sip_amount
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    GROUP BY strftime('%Y', transaction_date)
)
SELECT
    year,
    ROUND(total_sip_amount, 2) AS total_sip_amount,
    ROUND(
        (
            total_sip_amount -
            LAG(total_sip_amount) OVER (ORDER BY year)
        ) * 100.0 /
        LAG(total_sip_amount) OVER (ORDER BY year),
        2
    ) AS yoy_growth_pct
FROM yearly_sip
ORDER BY year;
"""

sip_yoy_growth = pd.read_sql(query3, engine)

sip_yoy_growth

,year,total_sip_amount,yoy_growth_pct
0,2024,153233052.0,NaN
1,2025,64000439.0,-58.23


#Query 4 — Transactions by State

In [ ]:
query4 = """
SELECT
    state,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transaction_amount DESC;
"""

transactions_by_state = pd.read_sql(query4, engine)

transactions_by_state

,state,total_transactions,total_transaction_amount
0,Punjab,2965,315780459.0
1,Tamil Nadu,2806,315177237.0
2,Madhya Pradesh,2931,308312493.0
3,Rajasthan,2577,298645822.0
4,Gujarat,2780,298358940.0
5,West Bengal,2748,297182514.0
6,Telangana,2718,290219284.0
7,Delhi,2677,289633404.0
8,Uttar Pradesh,2695,285368873.0
9,Haryana,2736,279634354.0


#Query 5 — Funds with Expense Ratio Below 1%

In [ ]:
query5 = """
SELECT
    scheme_name,
    fund_house,
    category,
    expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1
ORDER BY expense_ratio_pct ASC;
"""

funds_low_expense = pd.read_sql(query5, engine)

funds_low_expense

,scheme_name,fund_house,category,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Debt,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Debt,0.56
2,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Debt,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,0.66
4,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Equity,0.72
5,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Debt,0.74
7,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,Equity,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Debt,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Equity,0.78


#Query 6 — Top 5 Funds by 5-Year Return

In [ ]:
query6 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_5yr_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.return_5yr_pct IS NOT NULL
ORDER BY p.return_5yr_pct DESC
LIMIT 5;
"""

top_5_returns = pd.read_sql(query6, engine)

top_5_returns

,scheme_name,fund_house,return_5yr_pct
0,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,23.80
1,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,22.62
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,21.88
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,21.82
4,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,20.67


#Query 7: Transaction Amount by Transaction Type.

In [ ]:
query7 = """
SELECT
    transaction_type,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_amount DESC;
"""

transaction_type_summary = pd.read_sql(query7, engine)

transaction_type_summary

,transaction_type,total_transactions,total_amount
0,Lumpsum,8095,2.059821e+09
1,Redemption,4967,1.244525e+09
2,SIP,19716,2.172335e+08


#Query 8 — Top 5 Fund Houses by AUM

In [ ]:
query8 = """
SELECT
    fund_house,
    ROUND(SUM(aum_crore), 2) AS total_aum_crore,
    SUM(num_schemes) AS total_schemes
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum_crore DESC
LIMIT 5;
"""

top_fund_houses_aum = pd.read_sql(query8, engine)

top_fund_houses_aum

,fund_house,total_aum_crore,total_schemes
0,SBI Mutual Fund,8491000.0,1674
1,ICICI Prudential MF,6293000.0,1944
2,HDFC Mutual Fund,5732000.0,1755
3,Nippon India MF,3909000.0,1593
4,Kotak Mahindra MF,3502000.0,1512


#Query 9 — Average Transaction Amount by Payment Mode

In [ ]:
query9 = """
SELECT
    payment_mode,
    COUNT(*) AS total_transactions,
    ROUND(AVG(amount_inr), 2) AS average_transaction_amount
FROM fact_transactions
GROUP BY payment_mode
ORDER BY average_transaction_amount DESC;
"""

payment_mode_analysis = pd.read_sql(query9, engine)

payment_mode_analysis

,payment_mode,total_transactions,average_transaction_amount
0,UPI,8154,108933.16
1,Cheque,8228,108436.95
2,Net Banking,8250,108302.14
3,Mandate,8146,104054.45


#Query 10 — Top 5 Funds by Sharpe Ratio

In [ ]:
query10 = """
SELECT
    f.scheme_name,
    f.fund_house,
    p.sharpe_ratio,
    p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.sharpe_ratio IS NOT NULL
ORDER BY p.sharpe_ratio DESC
LIMIT 5;
"""

top_sharpe_funds = pd.read_sql(query10, engine)

top_sharpe_funds

,scheme_name,fund_house,sharpe_ratio,return_3yr_pct
0,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,7.68,7.68
1,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,6.18,6.18
2,ABSL Liquid Fund - Regular - Growth,Aditya Birla Sun Life MF,5.14,5.14
3,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,1.84,7.37
4,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,1.52,6.07


#schema.sql

In [ ]:
schema_content = """
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date TEXT,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER
);

CREATE TABLE fact_nav (
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    PRIMARY KEY (amfi_code, date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE fact_transactions (
    investor_id INTEGER,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE fact_performance (
    amfi_code INTEGER PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating REAL,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE fact_aum (
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER,
    PRIMARY KEY (date, fund_house),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);
"""

with open("schema.sql", "w") as file:
    file.write(schema_content)

print("schema.sql created successfully!")

schema.sql created successfully!


#queries.sql

In [ ]:
queries_content = """
-- Query 1: Top 5 Funds by AUM
SELECT
    f.scheme_name,
    f.fund_house,
    p.aum_crore
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;


-- Query 2: Average NAV per Month
SELECT
    strftime('%Y-%m', date) AS month,
    ROUND(AVG(nav), 2) AS average_nav
FROM fact_nav
GROUP BY strftime('%Y-%m', date)
ORDER BY month;


-- Query 3: SIP Year-over-Year Growth
WITH yearly_sip AS (
    SELECT
        strftime('%Y', transaction_date) AS year,
        SUM(amount_inr) AS total_sip_amount
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    GROUP BY strftime('%Y', transaction_date)
)
SELECT
    year,
    ROUND(total_sip_amount, 2) AS total_sip_amount,
    ROUND(
        (
            total_sip_amount -
            LAG(total_sip_amount) OVER (ORDER BY year)
        ) * 100.0 /
        LAG(total_sip_amount) OVER (ORDER BY year),
        2
    ) AS yoy_growth_pct
FROM yearly_sip
ORDER BY year;


-- Query 4: Transactions by State
SELECT
    state,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_transaction_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transaction_amount DESC;


-- Query 5: Funds with Expense Ratio Below 1%
SELECT
    scheme_name,
    fund_house,
    category,
    expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1
ORDER BY expense_ratio_pct ASC;


-- Query 6: Top 5 Funds by 5-Year Return
SELECT
    f.scheme_name,
    f.fund_house,
    p.return_5yr_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.return_5yr_pct IS NOT NULL
ORDER BY p.return_5yr_pct DESC
LIMIT 5;


-- Query 7: Transaction Amount by Transaction Type
SELECT
    transaction_type,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_amount DESC;


-- Query 8: Top 5 Fund Houses by AUM
SELECT
    fund_house,
    ROUND(SUM(aum_crore), 2) AS total_aum_crore,
    SUM(num_schemes) AS total_schemes
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum_crore DESC
LIMIT 5;


-- Query 9: Average Transaction Amount by Payment Mode
SELECT
    payment_mode,
    COUNT(*) AS total_transactions,
    ROUND(AVG(amount_inr), 2) AS average_transaction_amount
FROM fact_transactions
GROUP BY payment_mode
ORDER BY average_transaction_amount DESC;


-- Query 10: Top 5 Funds by Sharpe Ratio
SELECT
    f.scheme_name,
    f.fund_house,
    p.sharpe_ratio,
    p.return_3yr_pct
FROM fact_performance p
JOIN dim_fund f
    ON p.amfi_code = f.amfi_code
WHERE p.sharpe_ratio IS NOT NULL
ORDER BY p.sharpe_ratio DESC
LIMIT 5;
"""

with open("queries.sql", "w") as file:
    file.write(queries_content)

print("queries.sql created successfully!")

queries.sql created successfully!


#data_dictionary.md

In [ ]:
datasets = {
    "investor_transactions": df_transactions,
    "nav_history": df_nav,
    "fund_master": df_fund,
    "scheme_performance": df_performance,
    "aum": df_aum,
    "category_inflows": df_inflow,
    "industry_folio_count": df_folios,
    "benchmark_indices": df_index,
    "portfolio_holdings": df_holdings,
    "monthly_sip_inflows": df_monthlySIP
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name.upper())
    print(f"{'='*50}")
    print(df.dtypes)


INVESTOR_TRANSACTIONS
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object

NAV_HISTORY
amfi_code      int64
date          object
nav          float64
dtype: object

FUND_MASTER
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

SCHEM

In [ ]:
data_dictionary = """
# Data Dictionary

This document describes the cleaned datasets used in the Bluestock Mutual Fund Analytics project.

**Source Reference:** All datasets were provided by the company as raw source datasets.

---

# 1. Investor Transactions

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| investor_id | object | Unique identifier for an investor. | Company-provided raw dataset |
| transaction_date | object | Date on which the mutual fund transaction occurred. | Company-provided raw dataset |
| amfi_code | int64 | Unique AMFI scheme code identifying the mutual fund scheme. | Company-provided raw dataset |
| transaction_type | object | Type of transaction, such as SIP, Lumpsum, or Redemption. | Company-provided raw dataset |
| amount_inr | int64 | Transaction amount in Indian Rupees. | Company-provided raw dataset |
| state | object | State of the investor. | Company-provided raw dataset |
| city | object | City of the investor. | Company-provided raw dataset |
| city_tier | object | Classification of the investor's city by tier. | Company-provided raw dataset |
| age_group | object | Age category of the investor. | Company-provided raw dataset |
| gender | object | Gender of the investor. | Company-provided raw dataset |
| annual_income_lakh | float64 | Annual income of the investor measured in lakhs of INR. | Company-provided raw dataset |
| payment_mode | object | Payment method used for the transaction. | Company-provided raw dataset |
| kyc_status | object | KYC verification status of the investor. | Company-provided raw dataset |

---

# 2. NAV History

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | int64 | Unique AMFI scheme code. | Company-provided raw dataset |
| date | object | Date for which NAV is recorded. | Company-provided raw dataset |
| nav | float64 | Net Asset Value of the mutual fund scheme. | Company-provided raw dataset |

---

# 3. Fund Master

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | int64 | Unique AMFI scheme code. | Company-provided raw dataset |
| fund_house | object | Mutual fund company managing the scheme. | Company-provided raw dataset |
| scheme_name | object | Name of the mutual fund scheme. | Company-provided raw dataset |
| category | object | Broad investment category of the scheme. | Company-provided raw dataset |
| sub_category | object | Specific sub-category of the mutual fund scheme. | Company-provided raw dataset |
| plan | object | Scheme plan type. | Company-provided raw dataset |
| launch_date | object | Date on which the scheme was launched. | Company-provided raw dataset |
| benchmark | object | Benchmark index used to evaluate scheme performance. | Company-provided raw dataset |
| expense_ratio_pct | float64 | Annual fund management expenses expressed as a percentage. | Company-provided raw dataset |
| exit_load_pct | float64 | Fee charged when investors redeem before specified conditions. | Company-provided raw dataset |
| min_sip_amount | int64 | Minimum investment amount allowed through SIP. | Company-provided raw dataset |
| min_lumpsum_amount | int64 | Minimum one-time investment amount allowed. | Company-provided raw dataset |
| fund_manager | object | Name of the fund manager responsible for managing the scheme. | Company-provided raw dataset |
| risk_category | object | Risk classification assigned to the mutual fund scheme. | Company-provided raw dataset |
| sebi_category_code | object | SEBI classification code for the mutual fund scheme. | Company-provided raw dataset |

---

# 4. Scheme Performance

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | int64 | Unique AMFI scheme code. | Company-provided raw dataset |
| scheme_name | object | Name of the mutual fund scheme. | Company-provided raw dataset |
| fund_house | object | Mutual fund company managing the scheme. | Company-provided raw dataset |
| category | object | Broad investment category. | Company-provided raw dataset |
| plan | object | Scheme plan type. | Company-provided raw dataset |
| return_1yr_pct | float64 | One-year return percentage. | Company-provided raw dataset |
| return_3yr_pct | float64 | Three-year return percentage. | Company-provided raw dataset |
| return_5yr_pct | float64 | Five-year return percentage. | Company-provided raw dataset |
| benchmark_3yr_pct | float64 | Three-year return percentage of the benchmark. | Company-provided raw dataset |
| alpha | float64 | Measure of excess return compared with the benchmark. | Company-provided raw dataset |
| beta | float64 | Measure of the scheme's sensitivity to market movements. | Company-provided raw dataset |
| sharpe_ratio | float64 | Risk-adjusted return measurement. | Company-provided raw dataset |
| sortino_ratio | float64 | Risk-adjusted return measurement considering downside risk. | Company-provided raw dataset |
| std_dev_ann_pct | float64 | Annualized standard deviation representing investment volatility. | Company-provided raw dataset |
| max_drawdown_pct | float64 | Maximum percentage decline from a peak value. | Company-provided raw dataset |
| aum_crore | int64 | Assets Under Management measured in crores of INR. | Company-provided raw dataset |
| expense_ratio_pct | float64 | Fund management expenses expressed as a percentage. | Company-provided raw dataset |
| morningstar_rating | int64 | Morningstar rating assigned to the scheme. | Company-provided raw dataset |
| risk_grade | object | Risk grade assigned to the scheme. | Company-provided raw dataset |

---

# 5. AUM

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| date | object | Date associated with the AUM record. | Company-provided raw dataset |
| fund_house | object | Mutual fund company. | Company-provided raw dataset |
| aum_lakh_crore | float64 | Assets Under Management measured in lakh crores. | Company-provided raw dataset |
| aum_crore | int64 | Assets Under Management measured in crores. | Company-provided raw dataset |
| num_schemes | int64 | Number of schemes managed by the fund house. | Company-provided raw dataset |

---

# 6. Category Inflows

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | object | Month associated with the inflow data. | Company-provided raw dataset |
| category | object | Mutual fund investment category. | Company-provided raw dataset |
| net_inflow_crore | float64 | Net inflow into the category measured in crores of INR. | Company-provided raw dataset |

---

# 7. Industry Folio Count

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | object | Month associated with the folio count. | Company-provided raw dataset |
| total_folios_crore | float64 | Total mutual fund folios measured in crores. | Company-provided raw dataset |
| equity_folios_crore | float64 | Number of equity mutual fund folios measured in crores. | Company-provided raw dataset |
| debt_folios_crore | float64 | Number of debt mutual fund folios measured in crores. | Company-provided raw dataset |
| hybrid_folios_crore | float64 | Number of hybrid mutual fund folios measured in crores. | Company-provided raw dataset |
| others_folios_crore | float64 | Number of folios in other categories measured in crores. | Company-provided raw dataset |

---

# 8. Benchmark Indices

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| date | object | Date of the benchmark index observation. | Company-provided raw dataset |
| index_name | object | Name of the benchmark market index. | Company-provided raw dataset |
| close_value | float64 | Closing value of the benchmark index. | Company-provided raw dataset |

---

# 9. Portfolio Holdings

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| amfi_code | int64 | Unique AMFI scheme code. | Company-provided raw dataset |
| stock_symbol | object | Stock ticker or trading symbol of the holding. | Company-provided raw dataset |
| stock_name | object | Name of the company or security held. | Company-provided raw dataset |
| sector | object | Sector to which the holding belongs. | Company-provided raw dataset |
| weight_pct | float64 | Percentage weight of the holding in the portfolio. | Company-provided raw dataset |
| market_value_cr | float64 | Market value of the holding measured in crores. | Company-provided raw dataset |
| current_price_inr | float64 | Current market price of the holding in INR. | Company-provided raw dataset |
| portfolio_date | object | Date of the portfolio holdings snapshot. | Company-provided raw dataset |

---

# 10. Monthly SIP Inflows

| Column | Data Type | Business Definition | Source |
|---|---|---|---|
| month | object | Month associated with SIP data. | Company-provided raw dataset |
| sip_inflow_crore | int64 | Total SIP inflow measured in crores of INR. | Company-provided raw dataset |
| active_sip_accounts_crore | float64 | Number of active SIP accounts measured in crores. | Company-provided raw dataset |
| new_sip_accounts_lakh | float64 | Number of new SIP accounts measured in lakhs. | Company-provided raw dataset |
| sip_aum_lakh_crore | float64 | SIP Assets Under Management measured in lakh crores. | Company-provided raw dataset |
| yoy_growth_pct | float64 | Year-over-year growth percentage. | Company-provided raw dataset |
"""

with open("data_dictionary.md", "w", encoding="utf-8") as file:
    file.write(data_dictionary)

print("data_dictionary.md created successfully!")

data_dictionary.md created successfully!
